In [1]:
import pandas as pd
import numpy as np

import geopandas as gpd
from geopandas.tools import geocode
from tqdm import tqdm   # progress bar for loops

from pyproj import CRS
from tqdm import tqdm
import json

import warnings
warnings.filterwarnings('ignore') 

# Google Earth Engine (pip install earthengine-api)

import geemap, ee

In [2]:
# import tailored function to compute the data at Block level (Manzana)

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # Repo root
from scr.utils import *


In [3]:
try:
        ee.Initialize()
        
except Exception as e:
    
        ee.Authenticate()
        ee.Initialize()

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


# Night Light Density - VIIRS (2013-present) 

In [4]:
# Selecting 2017

viirs = (ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")
         .filterDate('2017-01-01','2017-12-31')
         .select('avg_rad'))

In [5]:
blocks_peru = gpd.read_file("../data/raw_data/Manzana_2017/MANZANA.shp", 
                          crs = "EPSG:4326")

blocks_peru.columns

Index(['IDMANZANA', 'CODDPTO', 'CODPROV', 'CODDIST', 'CODZONA', 'SUFZONA',
       'CODMZNA', 'SUFMZNA', 'UBIGEO', 'CODCCPP', 'DEPARTAMEN', 'PROVINCIA',
       'DISTRITO', 'NOMCCCPP', 'IDCCPP', 'Shape_STAr', 'Shape_STLe',
       'geometry'],
      dtype='str')

In [6]:
blocks_peru.shape

(485696, 18)

In [7]:
blocks_peru.IDMANZANA.is_unique

True

### Load a dataset at district level and dummy for urban area 

In [8]:
districts_infoarea = pd.read_stata("../data/raw_data/Census2017_district.dta")


In [9]:
districts_urban = districts_infoarea[districts_infoarea['area'] == 1].copy()

In [10]:
districts_urban.shape

(1180, 2)

In [11]:
blocks_peru_urban = blocks_peru.merge(districts_urban, 
                                      left_on='UBIGEO',
                                      right_on='ubigeo',
                                      how='inner')

blocks_peru_urban = gpd.GeoDataFrame(blocks_peru_urban, 
                                     geometry='geometry', 
                                     crs="EPSG:4326")

In [12]:
blocks_peru_urban.shape

(422828, 20)

In [13]:
json_grids = shp_to_ee_fmt(blocks_peru_urban)    # Geopandas to Json file

output_dir = Path(r"..\data\temporal_data\NLD")
checkpoint_every = 5000

batch_data = {}
m = 1

for i in tqdm(range(235000, len(json_grids))):   # Loop to get night-light and transformation by each geopolitical geometry

    geometry = ee.Geometry.MultiPolygon(json_grids[i]['geometry']['coordinates'])
    reduced_cities = viirs.map(lambda x: reduce_mean(x,geometry,'light_mean','avg_rad'))
    nested_list = reduced_cities.reduceColumns(ee.Reducer.toList(2), ['date','light_mean']).values().get(0)
    df = pd.DataFrame(nested_list.getInfo(), columns=['date','light_mean'])
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = pd.DatetimeIndex(df['date']).year
    
    df['IDMANZANA'] = json_grids[i]['properties']["IDMANZANA"]
    df['UBIGEO'] = json_grids[i]['properties']["UBIGEO"]
    df['area'] = json_grids[i]['properties']["area"]
    
    batch_data[i] = df.groupby(['IDMANZANA','UBIGEO','area','year'], 
                               as_index=False)['light_mean'].mean()   # collapse from monthly to annual

    if (i + 1) % checkpoint_every == 0 or i == len(json_grids) - 1:   # save batch size  5000
        
        batch_start = i - len(batch_data) + 1
        pd.concat(batch_data.values(), ignore_index=True).to_stata(
            output_dir / f"NLD_block_{batch_start}_{i}.dta", write_index=False)
        print(f"Saved {m*checkpoint_every}")
        batch_data = {}
        m+=1


    

  3%|▎         | 5001/187828 [14:09<13:56:18,  3.64it/s]

Saved 5000


  5%|▌         | 10000/187828 [28:32<16:31:13,  2.99it/s]

Saved 10000


  8%|▊         | 15000/187828 [43:02<17:30:56,  2.74it/s] 

...

Saved 180000


 98%|█████████▊| 185000/187828 [10:47:17<27:24,  1.72it/s]   

Saved 185000


100%|██████████| 187828/187828 [11:02:39<00:00,  4.72it/s]  

Saved 190000


## II. Altitude

In [ ]:
srtm = ee.Image("CGIAR/SRTM90_V4")

In [ ]:
json_grids = shp_to_ee_fmt(blocks_peru_urban)    # Geopandas to Json file

output_dir = Path(r"..\data\temporal_data\GoogleEarth")
checkpoint_every = 5000

batch_data = {}
m = 1

for i in tqdm(range(len(json_grids))):   # Loop to get elevation by each geopolitical geometry

    geometry = ee.Geometry.MultiPolygon(json_grids[i]['geometry']['coordinates'])
    output = reduce_mean(srtm, geometry, 'elevation_mean', 'elevation')

    batch_data[i] = pd.DataFrame([{
        'IDMANZANA': json_grids[i]['properties']["IDMANZANA"],
        'UBIGEO': json_grids[i]['properties']["UBIGEO"],
        'area': json_grids[i]['properties']["area"],
        'elevation_mean': output.get('elevation_mean').getInfo(),
    }])

    if (i + 1) % checkpoint_every == 0 or i == len(json_grids) - 1:   # save batch size 5000

        batch_start = i - len(batch_data) + 1
        pd.concat(batch_data.values(), ignore_index=True).to_stata(
            output_dir / f"Elevation_block_{batch_start}_{i}.dta", write_index=False)
        print(f"Saved {m*checkpoint_every}")
        batch_data = {}
        m += 1

## III. Temperature

In [ ]:
worldclime = ee.Image("WORLDCLIM/V1/BIO").select('bio01').multiply(0.1)   # Annual Mean Temperature (°C) = bio01 (bio12 is precipitation)

In [ ]:
json_grids = shp_to_ee_fmt(blocks_peru_urban)    # Geopandas to Json file

output_dir = Path(r"..\data\temporal_data\GoogleEarth")
checkpoint_every = 5000

batch_data = {}
m = 1

for i in tqdm(range(len(json_grids))):   # Loop to get temperature by each geopolitical geometry

    geometry = ee.Geometry.MultiPolygon(json_grids[i]['geometry']['coordinates'])
    output = reduce_mean(worldclime, geometry, 'temp_mean', 'bio01')

    batch_data[i] = pd.DataFrame([{
        'IDMANZANA': json_grids[i]['properties']["IDMANZANA"],
        'UBIGEO': json_grids[i]['properties']["UBIGEO"],
        'area': json_grids[i]['properties']["area"],
        'temp_mean': output.get('temp_mean').getInfo(),
    }])

    if (i + 1) % checkpoint_every == 0 or i == len(json_grids) - 1:   # save batch size 5000

        batch_start = i - len(batch_data) + 1
        pd.concat(batch_data.values(), ignore_index=True).to_stata(
            output_dir / f"Temperature_block_{batch_start}_{i}.dta", write_index=False)
        print(f"Saved {m*checkpoint_every}")
        batch_data = {}
        m += 1